# Segmentation des logements par clustering - Ames Housing

## Contexte

Ce notebook est independant de `01_prediction_prix_ames.ipynb` (prediction du prix de vente,
apprentissage supervise). Ici, l'objectif est different : faire ressortir des **groupes homogenes
de logements** a partir de leurs caracteristiques physiques (taille, age, qualite, confort), sans
utiliser le prix de vente. C'est une tache de **clustering** (apprentissage non supervise) : on ne
dit pas au modele quels groupes existent, on le laisse les decouvrir a partir de la structure des
donnees.

Interet metier : au lieu de categories figees (`Bldg_Type`, `House_Style`), on obtient une typologie
construite empiriquement sur la taille, l'age, la qualite et le confort du bien - utile par exemple
pour adapter un discours commercial ou un systeme de recommandation par profil d'acheteur.

On reprend le meme dataset source que le notebook de prediction de prix, avec un nettoyage et un
feature engineering minimal, propres a ce notebook (pas de dependance a l'autre fichier).

## 1. Imports et configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

pd.set_option("display.max_columns", 100)
plt.rcParams["figure.figsize"] = (9, 5)
sns.set_style("whitegrid")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Chargement des donnees

In [ ]:
df = pd.read_csv("datas/AmesHousing_full.csv")

# nettoyage des noms de colonnes : espaces -> underscore, plus simple a manipuler
df.columns = [c.strip().replace(" ", "_").replace("/", "_") for c in df.columns]

# colonnes identifiants techniques, sans valeur predictive -> on les vire
id_cols = [c for c in ["Unnamed:_0", "Order", "PID"] if c in df.columns]
df = df.drop(columns=id_cols)

print("Shape apres nettoyage identifiants :", df.shape)
df.head()

## 3. Nettoyage minimal

On ne reprend que ce qui est necessaire pour les variables utilisees dans le clustering (section 5) :
- Les memes outliers de surface extreme que dans le notebook de prix (recommandation officielle du
  createur du dataset), pour eviter qu'un ou deux logements atypiques n'ecrasent tout le clustering.
- Les valeurs manquantes des colonnes numeriques liees a un equipement absent (ex : `Garage_Area` si
  pas de garage) sont mises a 0, meme logique que dans le notebook de prix : un `NaN` ici signifie
  "pas d'equipement", pas une donnee perdue.

In [ ]:
before = df.shape[0]
df = df[~((df["Gr_Liv_Area"] > 4000) & (df["SalePrice"] < 300000))].reset_index(drop=True)
print(f"Lignes retirees (outliers) : {before - df.shape[0]}")

zero_cols = ["Bsmt_Full_Bath", "Bsmt_Half_Bath", "Garage_Cars", "Garage_Area", "Total_Bsmt_SF"]
for c in zero_cols:
    if c in df.columns:
        df[c] = df[c].fillna(0)

print("Nouveau shape :", df.shape)

## 4. Feature engineering

Memes variables derivees que dans le notebook de prix, car elles ont un sens metier fort pour
decrire un logement independamment de son prix :

- `Total_SF` : surface totale (sous-sol + rez-de-chaussee + etage).
- `House_Age` : age de la maison au moment de la vente (`Yr_Sold - Year_Built`).
- `Years_Since_Remod` : anciennete de la derniere renovation.
- `Total_Bathrooms` : nombre total de salles de bain (completes + moitie, sous-sol + etages).

In [ ]:
df["Total_SF"] = df["Total_Bsmt_SF"] + df["1st_Flr_SF"] + df["2nd_Flr_SF"]
df["House_Age"] = df["Yr_Sold"] - df["Year_Built"]
df["Years_Since_Remod"] = df["Yr_Sold"] - df["Year_Remod_Add"]
df["Total_Bathrooms"] = (df["Full_Bath"] + 0.5 * df["Half_Bath"]
                          + df["Bsmt_Full_Bath"] + 0.5 * df["Bsmt_Half_Bath"])

# securite : age negatif signifie une erreur de saisie (vente avant construction) -> clip a 0
df["House_Age"] = df["House_Age"].clip(lower=0)
df["Years_Since_Remod"] = df["Years_Since_Remod"].clip(lower=0)

new_feats = ["Total_SF", "House_Age", "Years_Since_Remod", "Total_Bathrooms"]
df[new_feats].describe()

## 5. Choix des variables de clustering

On se limite volontairement aux **caracteristiques structurelles** du bien (taille, age, qualite,
confort), et on exclut :
- `SalePrice` : le but est de segmenter les logements par ce qu'ils *sont*, pas par ce qu'ils
  *valent*. On reutilisera le prix seulement a la fin, pour verifier si les clusters trouves ont un
  sens economique.
- `Bldg_Type`, `House_Style`, `Neighborhood` : ce sont des categories deja existantes dans les
  donnees ; le clustering doit decouvrir une typologie a partir des caracteristiques brutes, pas
  simplement reencoder ces labels. On les reutilisera aussi apres coup pour interpreter les clusters
  obtenus.

In [ ]:
cluster_features = [
    "Total_SF", "Lot_Area", "House_Age", "Years_Since_Remod",
    "Overall_Qual", "Overall_Cond",
    "Bedroom_AbvGr", "TotRms_AbvGrd", "Total_Bathrooms",
    "Garage_Cars", "Garage_Area", "Fireplaces",
]

X_cluster = df[cluster_features].copy()

print("Valeurs manquantes dans les variables de clustering :", X_cluster.isna().sum().sum())
X_cluster.describe()

## 6. Standardisation

KMeans est un algorithme base sur la distance euclidienne : sans normalisation, une variable comme
`Lot_Area` (valeurs en dizaines de milliers) ecraserait completement une variable comme
`Overall_Qual` (echelle 1-10) dans le calcul des distances. On standardise donc toutes les variables
(moyenne 0, ecart-type 1) avant le clustering.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

## 7. Choix du nombre de clusters (k)

Deux methodes complementaires pour explorer le choix de `k` :
- **Methode du coude (elbow)** : on regarde l'inertie (somme des distances au carre entre chaque
  point et le centre de son cluster) en fonction de `k`. Elle diminue toujours quand `k` augmente ;
  on cherche le point ou le gain marginal ralentit nettement (le "coude").
- **Score de silhouette** : mesure a quel point chaque point est bien assigne a son cluster par
  rapport aux clusters voisins (entre -1 et 1, plus c'est haut mieux c'est).

Sur ce dataset, la silhouette est maximale a **k=2** : c'est un resultat attendu et peu utile en
pratique. Avec des variables continues (surface, age, qualite) qui varient de facon graduelle plutot
que de former des groupes vraiment separes, la silhouette a tendance a favoriser une coupure grossiere
(par exemple "petits/bas de gamme" vs "grands/haut de gamme") simplement parce que deux gros blocs sont
mecaniquement mieux separes que 9 groupes plus fins. Un k=2 ne dit rien d'exploitable pour un usage
metier (recommandation, segmentation commerciale).

On fixe donc volontairement **`k = 9`** : la coherence interne des clusters (silhouette) est
necessairement un peu plus faible qu'a k=2, mais on gagne une granularite suffisante pour distinguer
des profils de logements varies (du petit logement ancien a la grande maison recente haut de gamme).
C'est un arbitrage frequent en clustering exploratoire : la silhouette guide, mais ne doit pas dicter
mecaniquement le choix final quand l'objectif est une typologie actionnable plutot qu'un decoupage
statistiquement optimal.

In [ ]:
inertias = []
silhouettes = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(k_range), inertias, marker="o")
axes[0].axvline(9, color="red", linestyle="--", alpha=0.6, label="k retenu = 9")
axes[0].set_xlabel("Nombre de clusters (k)")
axes[0].set_ylabel("Inertie")
axes[0].set_title("Methode du coude")
axes[0].legend()

axes[1].plot(list(k_range), silhouettes, marker="o", color="darkorange")
axes[1].axvline(9, color="red", linestyle="--", alpha=0.6, label="k retenu = 9")
axes[1].set_xlabel("Nombre de clusters (k)")
axes[1].set_ylabel("Score de silhouette")
axes[1].set_title("Score de silhouette par k")
axes[1].legend()

plt.tight_layout()
plt.show()

silhouette_argmax = list(k_range)[int(np.argmax(silhouettes))]
sil_at_argmax = silhouettes[list(k_range).index(silhouette_argmax)]
sil_at_9 = silhouettes[list(k_range).index(9)]

print(f"Silhouette maximale : k={silhouette_argmax} (score={sil_at_argmax:.3f})")
print(f"Silhouette a k=9 (retenu) : {sil_at_9:.3f}")

best_k = 9
print(f"k retenu pour l'analyse : {best_k}")

## 8. Entrainement du KMeans final

On entraine le KMeans definitif avec `best_k = 9` clusters (choix fixe volontairement en section 7),
puis on rattache le numero de cluster a chaque ligne du dataframe pour pouvoir les analyser.

In [ ]:
kmeans_final = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
df["Cluster"] = kmeans_final.fit_predict(X_scaled)

print(f"Repartition des {best_k} clusters :")
df["Cluster"].value_counts().sort_index()

## 9. Visualisation en 2D (PCA)

Les donnees standardisees ont autant de dimensions que de variables choisies (12 ici), impossibles a
visualiser directement. On projette sur les 2 premieres composantes principales (PCA) uniquement
pour la visualisation - le clustering, lui, a ete fait sur les donnees completes, pas sur cette
projection.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(9, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df["Cluster"], cmap="tab10", alpha=0.6, s=15)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% de variance expliquee)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% de variance expliquee)")
plt.title(f"Projection PCA des {best_k} clusters de logements")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.show()

## 10. Profil des clusters (interpretation metier)

Pour donner un sens a chaque cluster, on regarde la moyenne de chaque variable structurelle par
cluster, puis on croise avec des variables non utilisees dans le clustering (`SalePrice`,
`Bldg_Type`, `House_Style`) pour verifier si la typologie obtenue est coherente avec la realite du
marche.

In [ ]:
cluster_profile = df.groupby("Cluster")[cluster_features].mean().round(1)
cluster_profile

In [ ]:
# validation externe : le clustering n'a vu ni SalePrice, ni Bldg_Type, ni House_Style
validation = df.groupby("Cluster").agg(
    nb_logements=("Cluster", "size"),
    prix_moyen=("SalePrice", "mean"),
    prix_median=("SalePrice", "median"),
).round(0)
validation

In [ ]:
print("Type de batiment dominant par cluster :")
pd.crosstab(df["Cluster"], df["Bldg_Type"])

In [ ]:
print("Style de maison dominant par cluster :")
pd.crosstab(df["Cluster"], df["House_Style"])

### Nommage automatique des clusters

Pour rendre les clusters exploitables (dashboard, discours commercial), on leur donne un libelle
descriptif plutot qu'un simple numero. Le libelle est construit **automatiquement** a partir du
profil de chaque cluster (pas de valeurs codees en dur) : on decoupe les 9 clusters en tertiles
selon leur taille moyenne (`Total_SF`), leur age moyen (`House_Age`) et leur qualite moyenne
(`Overall_Qual`), puis on combine ces trois axes en une phrase courte.

In [ ]:
def tertile_labels(series, labels):
    ranks = series.rank(method="first")
    return pd.qcut(ranks, 3, labels=labels)

size_lbl = tertile_labels(cluster_profile["Total_SF"], ["petit", "moyen", "grand"])
age_lbl = tertile_labels(cluster_profile["House_Age"], ["recent", "d'age intermediaire", "ancien"])
qual_lbl = tertile_labels(cluster_profile["Overall_Qual"], ["entree de gamme", "milieu de gamme", "haut de gamme"])

cluster_profile["Type_Logement"] = (
    size_lbl.astype(str).str.capitalize() + " logement " + age_lbl.astype(str)
    + " (" + qual_lbl.astype(str) + ")"
)

label_map = cluster_profile["Type_Logement"].to_dict()
df["Type_Logement"] = df["Cluster"].map(label_map)

cluster_profile[["Type_Logement"]]

In [ ]:
print("Nombre de logements par type :")
df["Type_Logement"].value_counts()

## 11. Synthese et limites du clustering

**Lecture attendue** : si la segmentation est pertinente, les clusters doivent se distinguer
nettement sur la taille (`Total_SF`), l'age (`House_Age`) et la qualite (`Overall_Qual`), et le prix
moyen par cluster (calcule apres coup, jamais utilise dans le clustering) doit suivre un ordre
logique - un signe que la typologie decouverte a un sens economique et pas seulement statistique.

**Limites de cette approche :**
- KMeans suppose des clusters de forme spherique et de taille comparable, ce qui n'est pas garanti
  sur des donnees immobilieres reelles (ex : quelques logements atypiques tres grands ou tres anciens
  qui tirent un cluster vers eux).
- Le choix de `k` est assume comme subjectif plutot que purement statistique : la silhouette est
  maximale a k=2 sur ce dataset (segmentation grossiere, peu actionable), mais on a retenu k=9 pour
  obtenir une typologie exploitable commercialement (`Type_Logement`, section 10) - un choix guide par
  l'usage metier, pas par l'optimum mathematique.
- Les variables categorielles comme `Bldg_Type` ou le quartier n'ont pas ete integrees directement
  dans le calcul de distance (elles servent uniquement a la validation a posteriori) - une piste
  d'amelioration serait d'utiliser K-Prototypes ou une distance de Gower pour combiner variables
  numeriques et categorielles dans un seul clustering.

## 12. Export du dataset enrichi

On sauvegarde le dataset avec les colonnes ajoutees par cette analyse (`Cluster`, `Type_Logement`,
et les variables derivees comme `Total_SF`/`House_Age`) pour pouvoir le reutiliser ailleurs
(dashboard, partage, autre notebook) sans avoir a rejouer tout le clustering.

In [ ]:
import os

os.makedirs("datas", exist_ok=True)
output_path = "datas/AmesHousing_clustered.csv"
df.to_csv(output_path, index=False)

print(f"Dataset enrichi exporte : {output_path}")
print(f"Shape : {df.shape[0]} lignes x {df.shape[1]} colonnes")